In [ ]:
import glob
import os
import random
import json
import pickle
import pandas as pd
import sqlite3
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
import rubin_sim.maf.db as db
from rubin_sim.maf.utils import m52snr
import rubin_sim.maf.metrics as metrics
import rubin_sim.maf.plots as plots

import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.metricBundles as metricBundles
from rubin_sim.utils import equatorialFromGalactic, hpid2RaDec, _healbin, healbin
#from rubin_sim.utils import galactic_from_equatorial, _healbin, healbin
from rubin_sim.utils import getPackageDir
from rubin_sim.maf.metrics import tgaps
from TgapsPercentMetric import TgapsPercentMetric
import healpy as hp
import rubin_sim.maf.metricBundles as metricBundles


In [ ]:
 def make_hist(metricValue, slicer, userPlotDict):
        """
        ripped from https://github.com/lsst/sims_maf/blob/master/python/lsst/sims/maf/plots/specialPlotters.py
        
        Parameters
        ----------
        metricValue : numpy.ma.MaskedArray
            Handles 'object' datatypes for the masked array.
        slicer : lsst.sims.maf.slicers
            Any MAF slicer.
        userPlotDict: dict
            Dictionary of plot parameters set by user (overrides default values).
            'metricReduce' (an lsst.sims.maf.metric) indicates how to marginalize the metric values
            calculated at each point to a single series of values over the sky.
            'histStyle' (True/False) indicates whether to plot the results as a step histogram (True)
            or as a series of values (False)
            'bins' (np.ndarray) sets the x values for the resulting plot and should generally match
            the bins used with the metric.
        fignum : int
            Matplotlib figure number to use (default = None, starts new figure).
        Returns
        -------
        int
           Matplotlib figure number used to create the plot.
        """
        plotDict = {'metricReduce': metrics.SumMetric(), 'histStyle': False}
        plotDict.update(userPlotDict)
        # Combine the metric values across all slicePoints.
        if not isinstance(plotDict['metricReduce'], metrics.BaseMetric):
            raise ValueError('Expected plotDict[metricReduce] to be a MAF metric object.')
        # Get the data type
        dt = metricValue.compressed()[0].dtype
        # Change an array of arrays (dtype=object) to a 2-d array of correct dtype
        mV = np.array(metricValue.compressed().tolist(), dtype=[('metricValue', dt)])
        # Make an array to hold the combined result
        finalHist = np.zeros(mV.shape[1], dtype=float)
        metric = plotDict['metricReduce']
        metric.colname = 'metricValue'
        # Loop over each bin and use the selected metric to combine the results
        for i in np.arange(finalHist.size):
            finalHist[i] = metric.run(mV[:, i])
        bins = plotDict['bins']
        if plotDict['histStyle']:
            leftedge = bins[:-1]
            rightedge = bins[1:]

            x = np.vstack([leftedge, rightedge]).T.flatten()
            y = np.vstack([finalHist, finalHist]).T.flatten()

        else:
            # Could use this to plot things like FFT
            x = bins[:-1]
            y = finalHist
            
        return x, y

In [ ]:
import glob

families = ["baseline", "retro", "rolling", "noroll", "presto", "good_seeing", "long_u", "bluer"]
fbss = ["fbs_2.0", "fbs_2.1", "fbs_3.0", "v3.3_v3.4"]

files = []
for family in families:
    family_files = []  # List of files for this family
    for fbs in fbss:
        # Search for files in the primary directories
        family_files += glob.glob(f'../shared/__DEPRECATED__/rubin_opsims/{fbs}/{family}/*10yrs.db')
        family_files += glob.glob(f'../shared/rubin_opsims_{fbs}/{family}/*10yrs.db')
    
    # Specifically add baseline_v3.5 and v3.6 from the ../Stuff/Cadences/ directory
    if family == "baseline":
        family_files += glob.glob(f'../Stuff/Cadences/baseline_v3.5_10yrs.db')
        family_files += glob.glob(f'../Stuff/Cadences/baseline_v3.6_10yrs.db')
    
    files.append((family, family_files))

# Display the counts and file names for each family and fbss combination
for family, file_list in files:
    print(f"Family {family}: {len(file_list)} files found")
    for file_name in file_list:
        print(file_name)


In [ ]:
summary_fam = defaultdict(dict)
#for family, fbs in zip(families[:1], fbss[:1]):
summary_fam = defaultdict(dict)

for family in families:
    for fbs in fbss:
        # Use glob to make a list of database files
        if fbs == "fbs_2.1":
            files = glob.glob(f'../shared/__DEPRECATED__/rubin_opsims/fbs_2.1/{family}/*_10yrs.db')
        else:
            files = glob.glob(f'../shared/__DEPRECATED__/rubin_opsims/{fbs}/{family}/*10yrs.db')
            files += glob.glob(f'../shared/rubin_opsims_{fbs}/{family}/*10yrs.db')

        # Specifically add baseline_v3.5 and v3.6 if family is baseline
        if family == "baseline":
            files += glob.glob(f'../Stuff/Cadences/baseline_v3.5_10yrs.db')
            files += glob.glob(f'../Stuff/Cadences/baseline_v3.6_10yrs.db')

        files.sort()

        # Get run names from file names
        run_names = [filename.split('/')[-1].replace('.db', '') for filename in files]

        results = []
        for filename, run_name in zip(files, run_names):
            opsdb = db.OpsimDatabase(filename)
            outDir = 'fbs_3.6_plot'
            resultsDb = db.ResultsDb(outDir=outDir)

            bundleList = []
            sql = ''
            metric = metrics.TgapsMetric(bins=np.logspace(-3.46, 3.54, 99), allGaps=False)
            slicer = slicers.HealpixSlicer(nside=64)
            summaryMetrics = [metrics.MedianMetric()]
            plotDict = {'bins': np.logspace(-3.46, 3.54, 99)}
            plotFuncs = [plots.SummaryHistogram()]

            bundle = metricBundles.MetricBundle(metric, slicer, sql, runName=run_name, plotDict=plotDict, plotFuncs=plotFuncs)

            bd = metricBundles.makeBundlesDictFromList([bundle])
            bg = metricBundles.MetricBundleGroup(bd, opsdb, outDir=outDir, resultsDb=resultsDb)
            bg.runAll()

            # Update the summary_fam dictionary
            metric_names = bd.keys()
            summary_fam[family][run_name] = defaultdict(dict)

            for metric_name in metric_names:
                if 'Tgaps_' in metric_name:
                    try:
                        x, y = make_hist(bd[metric_name].metricValues, slicer, {'bins': np.logspace(-3.46, 3.54, 99)})
                    except KeyError:
                        continue
                    summary_fam[family][run_name][metric_name] = {"dt_hist_x": x, "dt_hist_y": y}
                else:
                    summary_fam[family][run_name][metric_name] = bd[metric_name].summaryValues

# Save the results in a pickle file
with open('fbs_3.6_plot_histograms_2024.pickle', 'wb') as handle:
    pickle.dump(summary_fam, handle, protocol=pickle.HIGHEST_PROTOCOL)

        

        
# Save the results in a pkl file
#with open('histograms_1.7.pickle', 'wb') as handle:
with open('fbs_3.6_plot_histograms_2024.pickle', 'wb') as handle:
#with open('histograms_1.5.pickle', 'wb') as handle:
    pickle.dump(summary_fam, handle, protocol=pickle.HIGHEST_PROTOCOL) 
########################
# # # Adjusted file paths and cadences
# # families = ["baseline_v2.0_10yrs", "baseline_v2.1_10yrs", "baseline_v3.0_10yrs"]
# # folders = ["rubin_opsims_v2.1", "rubin_opsims_v2.0", "rubin_opsims_v3.0"]
# # fbss = ['*v2.0_10yrs.db', '*v2.1_10yrs.db', '*v3.0_10yrs.db']

# summary_fam = defaultdict(dict)



# families = ["baseline_v2.0_10yrs", "baseline_v2.1_10yrs", "baseline_v3.0_10yrs"]
# folders = ["rubin_opsims_v2.1", "rubin_opsims_v2.0", "rubin_opsims_v3.0"]
# fbss = ["fbs_2.1", "fbs_2.0", "fbs_3.0"]  # Assuming this is the list of fbs versions

# for family in families:
#     for fbs in fbss:
#         if fbs == "fbs_2.1":
#             if not family == "baseline":
#                 continue
#             files = glob.glob("/sims_maf/fbs_2.1/baseline/*_10yrs.db")
#         else:
#             files = glob.glob(f'/sims_maf/{fbs}/{family}/*_10yrs.db')
#             files.sort()

#         print(f"Family: {family}, FBS: {fbs}")
       

In [ ]:
b = {}
for fname in ['fbs_3.6_plot_histograms_2024.pickle']:
    with open(fname, 'rb') as handle:
        d = pickle.load(handle)
        for key in d.keys():
            b[key] = d[key]

In [ ]:
b

In [ ]:
import matplotlib
font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 16}
matplotlib.rc('font', **font)

your attempt:

In [ ]:
import matplotlib.pyplot as plt

# Part C
import matplotlib
font = {'family': 'sans-serif',
        'weight': 'bold',
        'size': 14}
matplotlib.rc('font', **font)

fig, (ax1, ax2) = plt.subplots(2, figsize=(12, 10))
fig.text(0.5, 0.04, 'Spacing between images (days)', ha='center')
fig.text(0.04, 0.5, 'Number of observation pairs relative to v2.0 baseline', va='center', rotation='vertical')

cadences_to_plot = [
    "baseline_v2.0_10yrs",
    "baseline_v2.1_10yrs",
    "baseline_v3.0_10yrs",
    "baseline_v3.2_10yrs",
    "baseline_v3.3_10yrs",
    "baseline_v3.4_10yrs",
    "baseline_v3.5_10yrs",
    "baseline_v3.6_10yrs",
    "baseline_retrofoot_v2.0_10yrs",
    "retro_baseline_v2.0_10yrs",
    "rolling_ns2_rw0.5_v2.0_10yrs",
    #"rolling_ns2_rw0.9_v2.0_10yrs",
    "rolling_ns3_rw0.5_v2.0_10yrs",
    #"rolling_ns3_rw0.9_v2.0_10yrs",
    "noroll_v2.0_10yrs.db",
    #"good_seeing_gsw0.0_v2.1_10yrs",
    "good_seeing_gsw50.0_v2.1_10yrs",
    #"good_seeing_u_gsw0.0_v2.1_10yrs",
    "good_seeing_u_gsw50.0_v2.1_10yrs",
    "long_u2_v2.0_10yrs",
    "long_u1_v2.0_10yrs",
    "bluer_indx0_v2.0_10yrs",
    "bluer_indx1_v2.0_10yrs"
]


# Plotting for ax1
for family in ["baseline", "retro", "rolling", "noroll", "good_seeing", "long_u", "bluer"]:
    for run in b[family].keys():
        if run in cadences_to_plot:
            if run == "baseline_v2.0_10yrs":
                continue
            metric = list(b[family][run].keys())[0]
            x = b[family][run][metric]['dt_hist_x']
            y = b[family][run][metric]['dt_hist_y'] / b["baseline"]["baseline_v2.0_10yrs"][
                "baseline_v2_0_10yrs_Tgaps_observationStartMJD_HEAL"]["dt_hist_y"]
            ax1.semilogy(x, y, ds='steps-post', label=run)

ax1.set_xscale('log')
ax1.legend(bbox_to_anchor=(1, 1), loc='upper left', ncol=1)

# Plotting for ax2
for family in ['presto']:
    for run in b[family].keys():
        if run.startswith("presto_gap") and run not in ["presto_gap1.5_mix_v2.0_10yrs",
                                                        "presto_gap2.0_mix_v2.0_10yrs",
                                                        "presto_gap2.5_mix_v2.0_10yrs",
                                                        "presto_gap3.0_mix_v2.0_10yrs",
                                                        "presto_gap3.5_mix_v2.0_10yrs",
                                                        "presto_gap4.0_mix_v2.0_10yrs"]:
            metric = list(b[family][run].keys())[0]
            x = b[family][run][metric]['dt_hist_x']
            y = b[family][run][metric]['dt_hist_y'] / b["baseline"]["baseline_v2.0_10yrs"][
                "baseline_v2_0_10yrs_Tgaps_observationStartMJD_HEAL"]["dt_hist_y"]
            ax2.semilogy(x, y, ds='steps-post', label=run)

ax2.set_xscale('log')
ax2.legend(bbox_to_anchor=(1, 0.7), loc='upper left', ncol=1)

plt.savefig('fbs_v3.6_plot_tgap_histogram.pdf', bbox_inches="tight")
plt.show()


3 subplots

In [ ]:
import matplotlib.pyplot as plt

# Part C
import matplotlib
font = {'family': 'sans-serif',
        'weight': 'bold',
        'size': 16}
matplotlib.rc('font', **font)

fig, (ax1, ax2, ax3) = plt.subplots(3, figsize=(12, 15))
fig.text(0.5, 0.04, 'Spacing between images (days)', ha='center')
fig.text(0.04, 0.5, 'Number of observation pairs relative to v2.0 baseline', va='center', rotation='vertical')

cadences_to_plot = [
    "baseline_v2.0_10yrs",
    "baseline_v2.1_10yrs",
    "baseline_v3.0_10yrs",
    "baseline_v3.2_10yrs",
    "baseline_v3.3_10yrs",
    "baseline_v3.4_10yrs",
    "baseline_v3.5_10yrs",
    "baseline_v3.6_10yrs",
    "baseline_retrofoot_v2.0_10yrs",
    "retro_baseline_v2.0_10yrs",
    "rolling_ns2_rw0.5_v2.0_10yrs",
    #"rolling_ns2_rw0.9_v2.0_10yrs",
    "rolling_ns3_rw0.5_v2.0_10yrs",
    #"rolling_ns3_rw0.9_v2.0_10yrs",
    "noroll_v2.0_10yrs.db",
    #"good_seeing_gsw0.0_v2.1_10yrs",
    "good_seeing_gsw50.0_v2.1_10yrs",
    #"good_seeing_u_gsw0.0_v2.1_10yrs",
    "good_seeing_u_gsw50.0_v2.1_10yrs",
    "long_u2_v2.0_10yrs",
    "long_u1_v2.0_10yrs",
    "bluer_indx0_v2.0_10yrs",
    "bluer_indx1_v2.0_10yrs"
]

# Plotting for ax1
for family in ["baseline"]:
    for run in b[family].keys():
        if run in cadences_to_plot:
            metric = list(b[family][run].keys())[0]
            x = b[family][run][metric]['dt_hist_x']
            y = b[family][run][metric]['dt_hist_y'] / b["baseline"]["baseline_v2.0_10yrs"][
                "baseline_v2_0_10yrs_Tgaps_observationStartMJD_HEAL"]["dt_hist_y"]
            ax1.semilogy(x, y, ds='steps-post', label=run)

ax1.set_xscale('log')
ax1.legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)

# Plotting for ax2
for family in ['presto']:
    for run in b[family].keys():
        if run.startswith("presto_gap") and run not in ["presto_gap1.5_mix_v2.0_10yrs",
                                                        "presto_gap2.0_mix_v2.0_10yrs",
                                                        "presto_gap2.5_mix_v2.0_10yrs",
                                                        "presto_gap3.0_mix_v2.0_10yrs",
                                                        "presto_gap3.5_mix_v2.0_10yrs",
                                                        "presto_gap4.0_mix_v2.0_10yrs"]:
            metric = list(b[family][run].keys())[0]
            x = b[family][run][metric]['dt_hist_x']
            y = b[family][run][metric]['dt_hist_y'] / b["baseline"]["baseline_v2.0_10yrs"][
                "baseline_v2_0_10yrs_Tgaps_observationStartMJD_HEAL"]["dt_hist_y"]
            ax2.semilogy(x, y, ds='steps-post', label=run)

ax2.set_xscale('log')
ax2.legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)

# Plotting for ax3
for family in ["retro", "rolling", "noroll", "good_seeing", "long_u", "bluer"]:
    for run in b[family].keys():
        if run in cadences_to_plot:
            metric = list(b[family][run].keys())[0]
            x = b[family][run][metric]['dt_hist_x']
            y = b[family][run][metric]['dt_hist_y'] / b["baseline"]["baseline_v2.0_10yrs"][
                "baseline_v2_0_10yrs_Tgaps_observationStartMJD_HEAL"]["dt_hist_y"]
            ax3.semilogy(x, y, ds='steps-post', label=run)

ax3.set_xscale('log')
ax3.legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)

plt.savefig('3plot_fbs_v3.6_plot_tgap_histogram.pdf', bbox_inches="tight")
plt.show()
